# Gold load (manual runner and verification)
Runs the Silver to Gold refresh once and then reconciles Gold against Silver.
In Step 4 the same `run_gold` call is added to the end of `driver_session`, so this notebook
is only needed for the first full build and for manual re-runs.

Prerequisite: `gold_setup` has been run and Silver is loaded.

In [0]:
%run ./common

In [0]:
%run ./gold_functions

In [0]:
import re
from datetime import date, datetime, timezone

for _w in ['batch_id', 'run_date', 'load_type', 'force_aggregates']:
    try:
        dbutils.widgets.remove(_w)
    except Exception:
        pass

dbutils.widgets.text('batch_id', 'gold_manual_001')
dbutils.widgets.text('run_date', datetime.now(timezone.utc).date().isoformat())
dbutils.widgets.dropdown('load_type', 'FULL', ['FULL', 'INCREMENTAL'])
dbutils.widgets.dropdown('force_aggregates', 'NO', ['NO', 'YES'])

print('Set the widgets, then run the cells below.')

In [0]:
batch_id = dbutils.widgets.get('batch_id').strip()
run_date = dbutils.widgets.get('run_date').strip()
load_type = dbutils.widgets.get('load_type').strip().upper()
force_aggregates = dbutils.widgets.get('force_aggregates') == 'YES'

if not re.fullmatch(r'[A-Za-z0-9_-]{1,100}', batch_id):
    raise ValueError('batch_id may only contain letters, numbers, _ and -')
if date.fromisoformat(run_date).isoformat() != run_date:
    raise ValueError('run_date must be YYYY-MM-DD')
if load_type not in ('FULL', 'INCREMENTAL'):
    raise ValueError('load_type must be FULL or INCREMENTAL')

summary = run_gold(batch_id, run_date, load_type, force_aggregates)

## Verification: Gold reconciles with Silver (read-only)

In [0]:
from pyspark.sql import functions as F

GOLD = f'{CATALOG}.gold'
SILVER = f'{CATALOG}.silver'


def n(table, where=None):
    df = spark.table(table)
    return (df.filter(where) if where else df).count()


problems = []


def expect(label, actual, expected):
    ok = actual == expected
    print(f"{'PASS' if ok else 'FAIL'}  {label}: {actual:,} (expected {expected:,})")
    if not ok:
        problems.append(label)


print('=== Row counts against Silver ===')
expect('fact_commits = silver.commits', n(f'{GOLD}.fact_commits'), n(f'{SILVER}.commits'))
expect('fact_issues = silver.issues', n(f'{GOLD}.fact_issues'), n(f'{SILVER}.issues'))
expect('fact_pull_requests = silver.pulls', n(f'{GOLD}.fact_pull_requests'), n(f'{SILVER}.pulls'))
expect('dim_repository = silver.repo_metadata', n(f'{GOLD}.dim_repository'), n(f'{SILVER}.repo_metadata'))

print('\n=== Aggregates add up to the facts ===')
daily = spark.table(f'{GOLD}.fact_repo_daily_activity')
totals = daily.agg(*[F.sum(c).alias(c) for c in
                     ['commits', 'issues_opened', 'issues_closed', 'prs_opened',
                      'prs_merged', 'prs_closed_unmerged']]).first()
expect('sum(commits) = fact_commits', int(totals['commits']), n(f'{GOLD}.fact_commits'))
expect('sum(issues_opened) = fact_issues', int(totals['issues_opened']), n(f'{GOLD}.fact_issues'))
expect('sum(issues_closed) = closed issues', int(totals['issues_closed']),
       n(f'{GOLD}.fact_issues', 'is_closed'))
expect('sum(prs_opened) = fact_pull_requests', int(totals['prs_opened']),
       n(f'{GOLD}.fact_pull_requests'))
expect('sum(prs_merged) = merged PRs', int(totals['prs_merged']),
       n(f'{GOLD}.fact_pull_requests', 'is_merged'))
expect('sum(prs_closed_unmerged) = closed, not merged', int(totals['prs_closed_unmerged']),
       n(f'{GOLD}.fact_pull_requests', 'closed_date_key <> -1 AND NOT is_merged'))

activity = spark.table(f'{GOLD}.fact_contributor_activity').agg(
    F.sum('commits').alias('c'), F.sum('issues_opened').alias('i'),
    F.sum('prs_opened').alias('p')).first()
expect('contributor activity commits', int(activity['c']),
       n(f'{GOLD}.fact_commits', 'contributor_key <> -1'))
expect('contributor activity issues', int(activity['i']),
       n(f'{GOLD}.fact_issues', 'contributor_key <> -1'))
expect('contributor activity PRs', int(activity['p']),
       n(f'{GOLD}.fact_pull_requests', 'contributor_key <> -1'))

In [0]:
print('=== No orphan foreign keys ===')
FOREIGN_KEYS = [
    ('fact_commits', 'repository_key', 'dim_repository', 'repository_key'),
    ('fact_commits', 'contributor_key', 'dim_contributor', 'contributor_key'),
    ('fact_commits', 'commit_date_key', 'dim_date', 'date_key'),
    ('fact_issues', 'repository_key', 'dim_repository', 'repository_key'),
    ('fact_issues', 'contributor_key', 'dim_contributor', 'contributor_key'),
    ('fact_issues', 'created_date_key', 'dim_date', 'date_key'),
    ('fact_issues', 'closed_date_key', 'dim_date', 'date_key'),
    ('fact_pull_requests', 'repository_key', 'dim_repository', 'repository_key'),
    ('fact_pull_requests', 'contributor_key', 'dim_contributor', 'contributor_key'),
    ('fact_pull_requests', 'created_date_key', 'dim_date', 'date_key'),
    ('fact_pull_requests', 'closed_date_key', 'dim_date', 'date_key'),
    ('fact_pull_requests', 'merged_date_key', 'dim_date', 'date_key'),
    ('fact_repo_daily_activity', 'repository_key', 'dim_repository', 'repository_key'),
    ('fact_repo_daily_activity', 'date_key', 'dim_date', 'date_key'),
    ('fact_contributor_activity', 'repository_key', 'dim_repository', 'repository_key'),
    ('fact_contributor_activity', 'contributor_key', 'dim_contributor', 'contributor_key'),
    ('fact_contributor_activity', 'date_key', 'dim_date', 'date_key'),
    ('fact_repo_snapshot', 'repository_key', 'dim_repository', 'repository_key'),
    ('fact_repo_snapshot', 'snapshot_date_key', 'dim_date', 'date_key'),
]
for fact, column, dim, dim_key in FOREIGN_KEYS:
    dim_df = spark.table(f'{GOLD}.{dim}').select(F.col(dim_key).alias('_k'))
    orphans = (spark.table(f'{GOLD}.{fact}')
               .join(F.broadcast(dim_df), F.col(column) == F.col('_k'), 'left_anti').count())
    expect(f'{fact}.{column} -> {dim}', orphans, 0)

print('\n=== No duplicate primary keys ===')
PRIMARY_KEYS = {
    'dim_repository': ['repository_key'],
    'dim_contributor': ['contributor_key'],
    'fact_commits': ['record_key'],
    'fact_issues': ['record_key'],
    'fact_pull_requests': ['record_key'],
    'fact_repo_daily_activity': ['repository_key', 'date_key'],
    'fact_contributor_activity': ['repository_key', 'contributor_key', 'date_key'],
    'fact_repo_snapshot': ['repository_key', 'snapshot_date_key'],
}
for table, keys in PRIMARY_KEYS.items():
    dupes = spark.table(f'{GOLD}.{table}').groupBy(*keys).count().filter('count > 1').count()
    expect(f'{table} duplicate keys', dupes, 0)

print()
if problems:
    raise AssertionError(f'{len(problems)} check(s) failed: {problems}')
print('All Gold checks passed')

## Look at the result

In [0]:
display(spark.sql(f'''
  SELECT r.repository_name, d.year_month,
         SUM(a.commits) AS commits, SUM(a.issues_opened) AS issues_opened,
         SUM(a.issues_closed) AS issues_closed, SUM(a.prs_opened) AS prs_opened,
         SUM(a.prs_merged) AS prs_merged
  FROM {GOLD}.fact_repo_daily_activity a
  JOIN {GOLD}.dim_repository r ON a.repository_key = r.repository_key
  JOIN {GOLD}.dim_date d       ON a.date_key = d.date_key
  GROUP BY r.repository_name, d.year_month
  ORDER BY r.repository_name, d.year_month
'''))

display(spark.sql(f'''
  SELECT layer, entity_type, status, start_time, end_time,
         unix_timestamp(end_time) - unix_timestamp(start_time) AS seconds,
         rows_processed, inserted_rows, updated_rows
  FROM {CATALOG}.ops.pipeline_execution_logs
  WHERE layer = 'GOLD' AND batch_id = '{batch_id}'
  ORDER BY start_time
'''))